# 🛠️ Dataset Prep: Dataset Split

## Cleanup Label Studio Labels

Label Studio exports prefix files with an 8-character hex ID (e.g. `0b2963ce-IMG_1634.txt`). The ID is stripped from image and label names before pairing. Source files are not renamed; only the split copies get clean names.

In [ ]:
import json
import random
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

# Relative to the notebook folder
source_images = Path("images")
source_labels = Path("labels")
output_dir = Path("split")

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}
LS_ID_PATTERN = re.compile(r"^[0-9a-fA-F]{8}-")


def strip_ls_id(stem):
    return LS_ID_PATTERN.sub("", stem, count=1)


def index_by_clean_stem(folder, exts, skip_names=()):
    index, collisions = {}, {}
    for path in sorted(folder.iterdir()):
        if not path.is_file() or path.suffix.lower() not in exts or path.name in skip_names:
            continue
        stem = strip_ls_id(path.stem)
        if stem in index:
            collisions.setdefault(stem, [index[stem].name]).append(path.name)
        else:
            index[stem] = path
    return index, collisions


images, image_collisions = index_by_clean_stem(source_images, IMAGE_EXTS)
labels, label_collisions = index_by_clean_stem(source_labels, {".txt"}, skip_names={"classes.txt"})

if image_collisions or label_collisions:
    for kind, found in (("image", image_collisions), ("label", label_collisions)):
        for stem, names in found.items():
            print(f"{kind} collision '{stem}': {names}")
    raise ValueError("Multiple files map to the same name after stripping Label Studio IDs.")

unlabeled_images = sorted(images.keys() - labels.keys())
orphan_labels = sorted(labels.keys() - images.keys())
pairs = [(stem, images[stem], labels.get(stem)) for stem in sorted(images)]

print(f"Images: {len(images)} | Labels: {len(labels)} | Matched pairs: {len(images) - len(unlabeled_images)}")
print(f"Images without labels (kept as background): {len(unlabeled_images)}")
for stem in unlabeled_images:
    print(f"  {images[stem].name}")
print(f"Labels without images (skipped): {len(orphan_labels)}")
for stem in orphan_labels:
    print(f"  {labels[stem].name}")

## Sort Data

Seeded train/val/test split. With `RANDOM_SEED = None` a seed is generated from the current UTC time (`YYYYMMDDHHMMSS`) and printed; paste it into `RANDOM_SEED` to reproduce the split. The seed and file assignments are saved to `split_manifest.json` in `output_dir`.

In [ ]:
RANDOM_SEED = None
SPLIT_RATIO = {"train": 0.70, "val": 0.15, "test": 0.15}

created_utc = datetime.now(timezone.utc)
if RANDOM_SEED is None:
    RANDOM_SEED = int(created_utc.strftime("%Y%m%d%H%M%S"))
    print(f"Generated RANDOM_SEED = {RANDOM_SEED}  (paste into RANDOM_SEED to reproduce)")
else:
    print(f"Using RANDOM_SEED = {RANDOM_SEED}")

shuffled = list(pairs)
random.Random(RANDOM_SEED).shuffle(shuffled)

total = len(shuffled)
counts = {name: round(total * ratio) for name, ratio in SPLIT_RATIO.items()}
if total >= 3:
    counts = {name: max(1, n) for name, n in counts.items()}
counts["train"] = total - counts["val"] - counts["test"]

assignments = {
    "train": shuffled[:counts["train"]],
    "val": shuffled[counts["train"]:counts["train"] + counts["val"]],
    "test": shuffled[counts["train"] + counts["val"]:],
}

for split in assignments:
    for kind in ("images", "labels"):
        split_dir = output_dir / kind / split
        if split_dir.is_dir() and any(split_dir.iterdir()):
            print(f"WARNING: {split_dir} already contains files; old files will be mixed with this split.")
        split_dir.mkdir(parents=True, exist_ok=True)

manifest_splits = {}
for split, items in assignments.items():
    records, list_lines = [], []
    for stem, img_src, lbl_src in items:
        img_dest = output_dir / "images" / split / f"{stem}{img_src.suffix}"
        shutil.copy2(img_src, img_dest)
        list_lines.append(str(img_dest.resolve()))
        lbl_name = None
        if lbl_src is not None:
            lbl_name = f"{stem}.txt"
            shutil.copy2(lbl_src, output_dir / "labels" / split / lbl_name)
        records.append({
            "stem": stem,
            "source_image": img_src.name,
            "image": img_dest.name,
            "source_label": lbl_src.name if lbl_src else None,
            "label": lbl_name,
        })
    manifest_splits[split] = records
    (output_dir / f"{split}.txt").write_text("\n".join(list_lines) + "\n", encoding="utf-8")

manifest = {
    "random_seed": RANDOM_SEED,
    "created_utc": created_utc.isoformat(),
    "split_ratio": SPLIT_RATIO,
    "source_images": str(source_images),
    "source_labels": str(source_labels),
    "counts": {split: len(items) for split, items in assignments.items()},
    "unlabeled_images": [images[s].name for s in unlabeled_images],
    "orphan_labels": [labels[s].name for s in orphan_labels],
    "splits": manifest_splits,
}
(output_dir / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Split counts: {manifest['counts']}")
print(f"Manifest: {output_dir / 'split_manifest.json'}")

Dataset successfully split into train, val, and test sets.
